# Virtualize NASA Earthdata — TEMPO L2 and ITS-LIVE

This notebook demonstrates how to create virtual (cloud-optimized) datasets from NASA Earthdata
collections without downloading science data.

## TEMPO L2 — ragged scanlines

TEMPO measures air quality in a 2-D swath scan with dimensions `(mirror_step, xtrack)`.  
Each granule covers one portion of a scan; the **last granule in a scan** may have fewer `mirror_step`
rows than the others (ragged edge).  The desired combined dataset has:

```
Dimensions:  (time, mirror_step, xtrack)
Coordinates: xtrack (1-D), time (1-D synthetic)
Variables:   time(time, mirror_step), lat(time, mirror_step, xtrack),
             lon(time, mirror_step, xtrack), no2(...), ...
```

The two-level concat strategy (from [VirtualiZarr issue #487](https://github.com/zarr-developers/VirtualiZarr/issues/487)):

- **Inner concat** (`mirror_step`): stitch granules within each scan, padding the ragged last granule.
- **Outer concat** (`time`): stack completed scans — no padding needed, all scans same shape.

```
concat_dim=["time", "mirror_step"]
                ↑              ↑
           outer (scans)  inner (granules within scan)
```

## ITS-LIVE — spatial mosaic

ITS-LIVE velocity granules cover different geographic tiles.  `mosaic_dims=["y","x"]` unions them
into a single spatial grid before the time-axis concat.


## 1. Authentication

`earthaccess.login()` tries (in order):
1. `EARTHDATA_USERNAME` / `EARTHDATA_PASSWORD` environment variables
2. `~/.netrc`
3. Interactive browser / prompt


In [ ]:
import earthaccess
from dask.distributed import Client

earthaccess.login()

# Extract the Bearer token so Dask workers can authenticate.
# Workers are separate processes and do not inherit the fsspec session;
# register the plugin so every worker configures fsspec auth before
# any compute task runs.
token = earthaccess.__auth__.token["access_token"]

client = Client(n_workers=8, threads_per_worker=1)
client.register_plugin(earthaccess.dask_worker_init(token))
print(client.dashboard_link)
print(earthaccess.__version__)


## 2. Search TEMPO L2 granules

Filter to a single product version (`V03`) and a short temporal window.
Mixed versions produce incompatible variable shapes or attributes — always filter.


In [ ]:
COLLECTION = "TEMPO_NO2_L2"
TEMPORAL   = ("2024-03-28", "2024-03-28")   # one day
VERSION    = "V03"

granules = earthaccess.search_data(
    short_name=COLLECTION,
    temporal=TEMPORAL,
    version=VERSION,
)
print(f"Found {len(granules)} granules")

## 3. Group granules by scan

TEMPO granule filenames encode the scan number (`S003G01`, `S003G02`, …).
We group granules by scan so the nested concat can align them correctly.

The result is a nested list of URL lists:
```
[
    [scan1_granule1, scan1_granule2, ...],   # inner list = one scan
    [scan2_granule1, scan2_granule2, ...],
    ...
]
```


In [ ]:
import re
from collections import defaultdict

def _scan_number(granule) -> int:
    """Extract scan number from TEMPO granule filename (S003 → 3)."""
    fname = granule["meta"]["native-id"]
    m = re.search(r"S(\d+)G", fname)
    return int(m.group(1)) if m else 0

# Group DataGranule objects by scan number.
# earthaccess.virtualize() accepts nested granule lists for two-level concat.
scan_map: dict[int, list] = defaultdict(list)
for g in granules:
    scan_map[_scan_number(g)].append(g)

nested_granules = [scan_map[k] for k in sorted(scan_map)]

print(f"{len(nested_granules)} scans")
for i, scan in enumerate(nested_granules[:3]):
    print(f"  scan {i}: {len(scan)} granules")


## 4. Preprocess: assign real UTC `time` coordinate

`xr.combine_nested` needs a scalar coordinate on the outer dimension so it can
label the combined dataset.  We parse each granule's `time_coverage_start` global
attribute into a `numpy.datetime64` and assign it as a scalar `time` coordinate.

This gives the final virtual dataset a real, selectable `time` axis
(e.g. `vds.sel(time='2024-03-28T14:00', method='nearest')`).


In [ ]:
import re
import numpy as np
import xarray as xr

def add_time(ds: xr.Dataset) -> xr.Dataset:
    """Assign a scalar UTC datetime64 'time' coordinate parsed from the granule filename.

    Global attrs (e.g. time_coverage_start) are only present on the root group;
    when opening a sub-group (product, geolocation) attrs are empty.  The
    granule filename always contains the timestamp as T<YYYYMMDDTHHMMSS>Z.

    The geolocation group contains a per-row 'time' variable (GPS seconds).
    We rename it to 'scan_time' to free up the 'time' name for the outer
    scan-level dimension coordinate.
    """
    if "time" in ds:
        ds = ds.rename({"time": "scan_time"})

    # Parse timestamp from filename: ...20240328T114338Z...
    source = ds.encoding.get("source", "")
    m = re.search(r'(\d{4})(\d{2})(\d{2})T(\d{2})(\d{2})(\d{2})Z', source)
    if m:
        ts_str = f"{m.group(1)}-{m.group(2)}-{m.group(3)}T{m.group(4)}:{m.group(5)}:{m.group(6)}"
        ts = np.datetime64(ts_str, 'ns')
    else:
        ts = np.datetime64('NaT', 'ns')

    return ds.expand_dims('time').assign_coords(time=[ts])


## 5. Virtualize — two-level nested concat

Key parameters:

| Parameter | Value | Why |
|---|---|---|
| `concat_dim` | `["time", "mirror_step"]` | outer = scans (real UTC timestamps), inner = granules within scan |
| `pad` | `"auto"` | pad ragged last granule of each scan to `max(mirror_step)` |
| `combine` | `"nested"` | required for multi-dim concat |
| `preprocess` | `add_time` | parses `time_coverage_start` → scalar `datetime64` coord |
| `loadable_variables` | `[]` | keep everything virtual; load lat/lon at analysis time |
| `group` | `"product"` | TEMPO group containing NO₂ variables |

Padding is applied **only at the inner `mirror_step` level**.  After inner concat all scans have the
same shape, so the outer `time` concat is clean with no padding.


In [ ]:
import time as _time

t0 = _time.perf_counter()

vds_product = earthaccess.virtualize(
    nested_granules,
    access="indirect",
    group="product",
    concat_dim=["time", "mirror_step"],  # outer=time (real UTC), inner=mirror_step
    pad="auto",
    preprocess=add_time,
    loadable_variables=[],
)

print(f"Virtualized in {_time.perf_counter() - t0:.2f} s")
print(f"Dims:       {dict(vds_product.sizes)}")
print(f"Data vars:  {list(vds_product.data_vars)}")
print(f"Coords:     {list(vds_product.coords)}")
vds_product


## 6. Add geolocation (lat, lon)

TEMPO stores `lat`, `lon` in the `geolocation` group.  Virtualize it separately with the same
two-level concat, then merge.


In [ ]:
t0 = _time.perf_counter()

vds_geo = earthaccess.virtualize(
    nested_granules,
    access="indirect",
    group="geolocation",
    concat_dim=["time", "mirror_step"],
    pad="auto",
    preprocess=add_time,
    # Inline lat/lon/scan_time so re-opening the store needs no network calls
    # for spatial masking.  TEMPO is geostationary — the pixel grid never
    # moves — so embedding the grid once is correct and saves ~25 MB of
    # per-chunk HTTP requests on every .compute().
    # "time" is the original name before preprocess renames it to "scan_time".
    loadable_variables=["time", "latitude", "longitude"],
)

print(f"Geolocation virtualized in {_time.perf_counter() - t0:.2f} s")
print(f"Dims:       {dict(vds_geo.sizes)}")
print(f"Data vars:  {list(vds_geo.data_vars)}")


In [ ]:
# Merge product + geolocation into one virtual dataset.
# Both groups share the same underlying dims (time, mirror_step, xtrack).
vds = xr.merge([vds_product, vds_geo], compat="override")

print(f"Merged dims: {dict(vds.sizes)}")
print(f"All variables: {sorted(vds.data_vars)}")
print()
# Expected structure:
#   Dimensions:  (time, mirror_step, xtrack)
#   Coordinates: time(time)  ← real UTC datetime64
#   Variables:   lat(time, mirror_step, xtrack)   ← virtual
#                lon(time, mirror_step, xtrack)    ← virtual
#                vertical_column_troposphere(time, mirror_step, xtrack)
#                ...
vds


## 7. Serialize to `./stores/tempo_no2.parquet`

Write the merged virtual dataset to a local kerchunk parquet store.
This is a **reference file only** — no actual data is downloaded.
The store can be reopened instantly with `earthaccess.open_virtual()`.


In [ ]:
import os

STORE_PATH = "./stores/tempo_no2.parquet"
os.makedirs("./stores", exist_ok=True)

t0 = _time.perf_counter()
vds.vz.to_kerchunk(STORE_PATH, format="parquet")
print(f"Written in {_time.perf_counter() - t0:.2f} s → {STORE_PATH}")
print(f"File size: {os.path.getsize(STORE_PATH) / 1024:.1f} KB")


### Re-open with `earthaccess.open_virtual()`

No network calls — just reads the local parquet reference file.
Data is only fetched from NASA servers when you call `.compute()`.


In [ ]:
ds = earthaccess.open_virtual(STORE_PATH, load=True)

print(f"Dims:      {dict(ds.sizes)}")
print(f"Coords:    {list(ds.coords)}")
print(f"Variables: {sorted(ds.data_vars)}")
ds


## 8. Analysis — bbox time series over a city

Load `latitude`/`longitude` once to build a spatial mask, then compute
the NO₂ time series.  Only the pixels inside the bbox are downloaded.

```
ds (time=98, mirror_step=1584, xtrack=2048)  ← virtual, no data yet
  ↓ load latitude/longitude  (one-time cost)
mask (time, mirror_step, xtrack)  bool
  ↓ where + mean over spatial dims
ts   (time=98,)               ← only bbox pixels fetched
```


In [ ]:
# --- Step 1: load lat/lon to build the spatial mask ----------------------
# latitude/longitude are (mirror_step, xtrack) — TEMPO is geostationary,
# the pixel grid does not move between scans.
lat = ds["latitude"].values   # (mirror_step, xtrack)
lon = ds["longitude"].values

# --- Step 2: define a city bounding box ----------------------------------
# Example: New York City
LAT_MIN, LAT_MAX = 40.5, 41.0
LON_MIN, LON_MAX = -74.3, -73.7

mask = (
    (lat >= LAT_MIN) & (lat <= LAT_MAX) &
    (lon >= LON_MIN) & (lon <= LON_MAX)
)  # (mirror_step, xtrack) bool — broadcasts over time automatically

print(f"Pixels in bbox: {mask.sum()} / {mask.size}")

# --- Step 3: compute time series -----------------------------------------
# xarray broadcasts the 2D mask over the time dimension automatically.
# Only the bbox pixels are fetched from the remote store.
no2 = ds["vertical_column_troposphere"]
ts = no2.where(mask).mean(["mirror_step", "xtrack"]).compute()

print(f"Time series shape: {ts.shape}")
print(ts.to_pandas())


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 4))
ts.plot(ax=ax, marker='o', linewidth=1.5)
ax.set_title(f"TEMPO NO₂ — NYC bbox ({LAT_MIN}–{LAT_MAX}°N, {LON_MIN}–{LON_MAX}°E)")
ax.set_ylabel("Vertical column [mol/m²]")
ax.set_xlabel("Time (UTC)")
plt.tight_layout()
plt.show()


## 9. (Optional) Start a Dask Distributed cluster

For large granule collections (hundreds of scans), wrap the virtualization in Dask.
`open_virtual_mfdataset` accepts `parallel="dask"` which uses `dask.delayed` per granule.
If a `distributed.Client` is active, tasks run on the cluster.


In [ ]:
# The Dask cluster started in cell 2 is already active.
# Add parallel="dask" to any earthaccess.virtualize() call to distribute
# the per-granule opens across the 8 workers:
#
# vds_product = earthaccess.virtualize(
#     nested_granules, ..., parallel="dask"
# )
#
# Scale up for large collections:
# client.cluster.scale(32)


### Icechunk alternative


In [ ]:
# icechunk alternative
# import icechunk
#
# ICE_PATH = "./stores/tempo_no2.icechunk"
# repo = icechunk.Repository.create(
#     icechunk.local_filesystem_storage(ICE_PATH),
#     config=icechunk.RepositoryConfig(inline_chunk_threshold_bytes=0),
# )
# session = repo.writable_session("main")
# vds.vz.to_icechunk(session.store)
# session.commit("virtualize_tempo_no2")
# print(f"icechunk store written to {ICE_PATH}")

## 10. ITS-LIVE spatial mosaic

ITS-LIVE granules cover different spatial tiles.  `mosaic_dims=["y","x"]` builds a union grid
from loaded coordinate arrays, placing each granule's chunks at the correct spatial offset.

Loadable coordinates (`y`, `x`) must be explicitly listed so the mosaic planner can read them.


In [ ]:
# ITS-LIVE example (uncomment and replace URLs with real granule URLs)
#
# from obstore.store import HTTPStore
#
# ITS_LIVE_BASE = "https://its-live-data.s3.amazonaws.com"
# store = HTTPStore(ITS_LIVE_BASE)
# registry = ObjectStoreRegistry({ITS_LIVE_BASE: store})
#
# tile_urls = [
#     "https://its-live-data.s3.amazonaws.com/.../ITS_LIVE_vel_EPSG3413_G0120_X-3250000_Y-550000.nc",
#     "https://its-live-data.s3.amazonaws.com/.../ITS_LIVE_vel_EPSG3413_G0120_X-3130000_Y-550000.nc",
# ]
#
# vds_itslive = open_virtual_mfdataset(
#     tile_urls,
#     registry=registry,
#     parser=HDFParser(),
#     combine="nested",
#     concat_dim="time",
#     mosaic_dims=["y", "x"],
#     loadable_variables=["y", "x", "time"],  # required for mosaic
#     pad="none",                              # no ragged edges in ITS-LIVE
# )
# print(f"ITS-LIVE mosaic dims: {dict(vds_itslive.sizes)}")

## Appendix: `earthaccess.virtualize` parameter reference for TEMPO

```python
earthaccess.virtualize(
    nested_granules,                       # nested list [[scan1_granules], [scan2_granules], ...]
    access="indirect",                     # "indirect" (HTTPS) or "direct" (S3)
    group="product",                       # HDF5 group to open
    parser="HDFParser",                    # or "DMRPPParser" (default), "NetCDF3Parser", ...
    concat_dim=["time", "mirror_step"],# [outer_dim, inner_dim]
    pad="auto",                            # pad ragged mirror_step within each scan
    preprocess=add_time,               # adds scalar time coord per granule
    loadable_variables=[],                 # [] = keep all virtual; None = load dim coords
    parallel=False,                        # False / "dask" / "lithops" / Executor subclass
)
```

**Why `concat_dim=["time", "mirror_step"]` and not `["mirror_step", "time"]`?**

`xr.combine_nested` (and `open_virtual_mfdataset` with `combine="nested"`) interprets `concat_dim[0]`
as the **outermost** nesting level and `concat_dim[-1]` as the innermost.  For TEMPO:

```
nested_granules = [[s1g1, s1g2, s1g3],   ← inner list (mirror_step concat)
                   [s2g1, s2g2, s2g3]]   ← outer list (time concat)

concat_dim      = ["time",            ← matches outer list
                   "mirror_step"]         ← matches inner list
```

Padding is applied **within each inner group before that group's concat**.
The outer `time` concat always sees uniform shapes — no padding needed there.

**`earthaccess.virtualize` vs `open_virtual_mfdataset` directly**

| Feature | `earthaccess.virtualize` | `open_virtual_mfdataset` directly |
|---|---|---|
| Authentication | handled automatically | must build `ObjectStoreRegistry` manually |
| Nested granule lists | yes (`list[list[DataGranule]]`) | yes (`list[list[str]]`) |
| DMR++ → HDFParser fallback | automatic | manual |
| `load=True` kerchunk round-trip | built-in | manual |

Use `open_virtual_mfdataset` directly when you already have URLs and want
full control; use `earthaccess.virtualize` when starting from search results.
